# Exploración inicial del dataset de proyectos
Caso BizOps · Onboarding

**Objetivo:** conocer los datos disponibles antes de diseñar un sistema que estime el riesgo de un proyecto al ingresar y ayude a priorizar 50 cupos mensuales de atención personalizada.

Fuentes locales: `project_dataset.csv` y `Caso Onboarding.docx`. Esta etapa explora y propone análisis; todavía no entrena un modelo ni automatiza decisiones.

**Cómo usarlo:** guarda este notebook junto al CSV y ejecuta las celdas en orden. También puedes ubicarlo en una subcarpeta de la carpeta del CSV. Requiere Python 3, pandas y numpy; para abrirlo y ejecutarlo necesitas Jupyter Notebook, JupyterLab o VS Code con soporte de notebooks. Si faltan dependencias: `python -m pip install pandas numpy jupyterlab`. Las tablas y gráficas ya tienen resultados guardados.


In [1]:
from pathlib import Path
from html import escape
import hashlib
import platform
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 65)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# Gráficas SVG embebidas: no requieren librerías gráficas ni conexión a internet.
class HTMLLocal:
    def __init__(self, contenido):
        self.contenido = contenido
    def _repr_html_(self):
        return self.contenido

def barras(serie, titulo, color='#2563eb'):
    serie = serie.dropna()
    alto = 55 + len(serie) * 30
    maximo = max(float(serie.max()), 1e-9)
    elementos = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 {alto}" role="img" aria-label="{escape(titulo)}" style="max-width:100%;background:white;font-family:Arial">',
                 f'<text x="12" y="24" font-size="17" font-weight="bold">{escape(titulo)}</text>']
    for i, (nombre, valor) in enumerate(serie.items()):
        y = 45 + i * 30
        ancho = max(0, float(valor) / maximo * 440)
        elementos.append(f'<text x="12" y="{y+15}" font-size="12">{escape(str(nombre))}</text>')
        elementos.append(f'<rect x="330" y="{y}" width="{ancho}" height="20" fill="{color}"/>')
        elementos.append(f'<text x="{780}" y="{y+15}" font-size="12">{float(valor):,.2f}</text>')
    display(HTMLLocal(''.join(elementos) + '</svg>'))

rutas = [Path.cwd() / 'project_dataset.csv', Path.cwd().parent / 'project_dataset.csv']
ruta = next((p for p in rutas if p.is_file()), None)
if ruta is None:
    raise FileNotFoundError('Ubica el notebook junto a project_dataset.csv o ajusta rutas.')

# Preservamos N/A y None como textos: su significado se revisa antes de limpiar.
df = pd.read_csv(ruta, keep_default_na=False)
print(f'Archivo: {ruta.name} | Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print('SHA-256:', hashlib.sha256(ruta.read_bytes()).hexdigest())
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')


Archivo: project_dataset.csv | Filas: 4,000 | Columnas: 51
SHA-256: 645c600b1a89b126341ec2c84ea0347580937d71df9727efd04a3a007ec0ede7
Python 3.12.14 | pandas 3.0.1 | numpy 2.3.5


## 1. Primeros cinco registros
Cada fila parece representar un proyecto identificado por `Project_ID`. Se muestran todas las columnas; la tabla requiere desplazamiento horizontal.


In [2]:
display(df.head(5))


,Project_ID,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,External_Dependencies_Count,Change_Request_Frequency,Project_Phase,Requirement_Stability,Team_Turnover_Rate,Vendor_Reliability_Score,Historical_Risk_Incidents,Communication_Frequency,Regulatory_Compliance_Level,Technology_Familiarity,Geographical_Distribution,Stakeholder_Engagement_Level,Schedule_Pressure,Budget_Utilization_Rate,Executive_Sponsorship,Funding_Source,Market_Volatility,Integration_Complexity,Resource_Availability,Priority_Level,Organizational_Change_Frequency,Cross_Functional_Dependencies,Previous_Delivery_Success_Rate,Technical_Debt_Level,Project_Manager_Experience,Org_Process_Maturity,Data_Security_Requirements,Key_Stakeholder_Availability,Tech_Environment_Stability,Contract_Type,Resource_Contention_Level,Industry_Volatility,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor,Risk_Level
0,PROJ_0001,Construction,32,"1,526,276.550",32,9.700,16,Waterfall,Senior,3,3,1.050,Planning,Moderate,0.160,0.840,2,1.900,Medium,Expert,4,Medium,0.000,0.820,Moderate,Government,0.550,2.660,0.980,Medium,0.290,6,0.800,0.000,Mid-level PM,Managed,Medium,Limited,N/A,Time & Materials,High,Extreme,First-time,Basic,Basic,Fully Colocated,Good,10,5,1.000,High
1,PROJ_0002,Manufacturing,2,"390,790.150",9,2.720,9,Kanban,Mixed,0,2,2.610,Execution,Moderate,0.420,0.790,2,2.650,High,Familiar,5,Excellent,0.000,0.760,Weak,External,0.290,2.450,0.950,Low,0.500,3,0.730,0.000,Mid-level PM,Optimizing,Low,Excellent,N/A,Cost-Plus,Low,Stable,Occasional,Advanced,Formal,Fully Remote,Poor,9,3,1.000,Low
2,PROJ_0003,Manufacturing,2,"246,674.760",6,2.040,7,Agile,Mixed,1,0,0.830,Initiation,Stable,0.550,0.890,2,1.870,Medium,Familiar,3,Excellent,0.000,1.170,Strong,Internal,0.150,5.570,0.790,High,2.710,2,0.910,0.000,Mid-level PM,Ad-hoc,Medium,Moderate,N/A,Cost-Plus,High,Stable,Regular,None,None,Hybrid,Good,5,1,1.000,Medium
3,PROJ_0004,IT,12,"1,427,830.630",17,7.540,16,Scrum,Mixed,0,5,2.420,Initiation,Moderate,0.330,0.840,1,1.360,Low,Familiar,5,Medium,0.100,0.730,Moderate,Mixed,0.740,7.490,0.520,Critical,0.740,5,0.710,0.430,Certified PM,Managed,Strict,Good,Mixed,Hybrid,High,Extreme,Strategic,Formal,Basic,Hybrid,Basic,12,6,1.100,High
4,PROJ_0005,Construction,24,"1,696,746.640",24,6.680,17,Hybrid,Junior,0,2,0.160,Execution,Stable,0.360,0.860,1,1.020,Critical,Expert,2,Low,0.000,0.600,Moderate,Government,0.190,1.640,0.580,Medium,0.560,6,0.830,0.000,Certified PM,Defined,Medium,Moderate,N/A,Cost-Plus,High,Moderate,Occasional,Basic,None,Partially Colocated,Basic,9,6,1.000,High


## 2. Estructura y columnas
El documento menciona 49 columnas, pero el CSV entregado contiene **51**: un identificador, 49 posibles variables explicativas y el resultado `Risk_Level`. El conteo es verificable; no asumimos que esa sea la explicación de la diferencia documental.


In [3]:
display(pd.DataFrame({
    'indicador': ['Proyectos', 'Columnas', 'Numéricas', 'Texto / categorías', 'IDs únicos'],
    'valor': [len(df), df.shape[1], len(df.select_dtypes(include='number').columns),
              len(df.select_dtypes(exclude='number').columns), df['Project_ID'].nunique()]
}))
df.info()
display(pd.DataFrame({'posición': range(1, len(df.columns)+1), 'columna': df.columns,
                      'tipo_python': df.dtypes.astype(str).values}))


,indicador,valor
0,Proyectos,4000
1,Columnas,51
2,Numéricas,25
3,Texto / categorías,26
4,IDs únicos,4000


<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 51 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Project_ID                       4000 non-null   str    
 1   Project_Type                     4000 non-null   str    
 2   Team_Size                        4000 non-null   int64  
 3   Project_Budget_USD               4000 non-null   float64
 4   Estimated_Timeline_Months        4000 non-null   int64  
 5   Complexity_Score                 4000 non-null   float64
 6   Stakeholder_Count                4000 non-null   int64  
 7   Methodology_Used                 4000 non-null   str    
 8   Team_Experience_Level            4000 non-null   str    
 9   Past_Similar_Projects            4000 non-null   int64  
 10  External_Dependencies_Count      4000 non-null   int64  
 11  Change_Request_Frequency         4000 non-null   float64
 12  Project_Phase                  

,posición,columna,tipo_python
0,1,Project_ID,str
1,2,Project_Type,str
2,3,Team_Size,int64
3,4,Project_Budget_USD,float64
4,5,Estimated_Timeline_Months,int64
5,6,Complexity_Score,float64
6,7,Stakeholder_Count,int64
7,8,Methodology_Used,str
8,9,Team_Experience_Level,str
9,10,Past_Similar_Projects,int64


### Diccionario interpretativo y disponibilidad al ingreso
Las descripciones se infieren de los nombres y valores; no se entregó un diccionario oficial. **Candidato intake** significa que podría pedirse al ingresar, pero debe confirmarse que el valor histórico corresponde a ese momento. **Histórico / validar** exige una fuente previa y un corte temporal. **Posterior / riesgo de fuga** se excluye provisionalmente de un modelo de día 1. Estas etiquetas son propuestas de trabajo, no hechos documentados.


In [4]:
descripciones = {'Project_ID': ('Identificador del proyecto', 'Identificador'), 'Project_Type': ('Tipo o sector del proyecto', 'Candidato intake'), 'Team_Size': ('Número de integrantes del equipo', 'Candidato intake'), 'Project_Budget_USD': ('Presupuesto del proyecto en USD', 'Candidato intake'), 'Estimated_Timeline_Months': ('Duración estimada en meses', 'Candidato intake'), 'Complexity_Score': ('Puntaje de complejidad; validar escala y quién lo asigna', 'Candidato intake'), 'Stakeholder_Count': ('Cantidad de stakeholders', 'Candidato intake'), 'Methodology_Used': ('Metodología de trabajo', 'Candidato intake'), 'Team_Experience_Level': ('Nivel de experiencia del equipo', 'Candidato intake'), 'Past_Similar_Projects': ('Cantidad de proyectos similares previos', 'Histórico / validar'), 'External_Dependencies_Count': ('Número de dependencias externas', 'Candidato intake'), 'Change_Request_Frequency': ('Frecuencia de solicitudes de cambio; unidad no documentada', 'Posterior / riesgo de fuga'), 'Project_Phase': ('Fase registrada del proyecto', 'Posterior / riesgo de fuga'), 'Requirement_Stability': ('Estabilidad de los requerimientos', 'Candidato intake'), 'Team_Turnover_Rate': ('Tasa de rotación del equipo; ventana temporal por validar', 'Posterior / riesgo de fuga'), 'Vendor_Reliability_Score': ('Confiabilidad del proveedor; escala por validar', 'Histórico / validar'), 'Historical_Risk_Incidents': ('Incidentes históricos; validar si preceden al proyecto', 'Histórico / validar'), 'Communication_Frequency': ('Frecuencia de comunicación; unidad y momento por validar', 'Posterior / riesgo de fuga'), 'Regulatory_Compliance_Level': ('Nivel de cumplimiento regulatorio; significado por validar', 'Candidato intake'), 'Technology_Familiarity': ('Familiaridad con la tecnología', 'Candidato intake'), 'Geographical_Distribution': ('Distribución geográfica; unidad por validar', 'Candidato intake'), 'Stakeholder_Engagement_Level': ('Nivel de participación de stakeholders', 'Posterior / riesgo de fuga'), 'Schedule_Pressure': ('Presión del cronograma; cálculo por validar', 'Posterior / riesgo de fuga'), 'Budget_Utilization_Rate': ('Proporción de utilización presupuestaria', 'Posterior / riesgo de fuga'), 'Executive_Sponsorship': ('Nivel de patrocinio ejecutivo', 'Candidato intake'), 'Funding_Source': ('Fuente de financiación', 'Candidato intake'), 'Market_Volatility': ('Volatilidad del mercado; escala y fecha por validar', 'Histórico / validar'), 'Integration_Complexity': ('Complejidad de integraciones', 'Candidato intake'), 'Resource_Availability': ('Disponibilidad de recursos; escala por validar', 'Candidato intake'), 'Priority_Level': ('Prioridad del proyecto; validar cómo se decidió', 'Histórico / validar'), 'Organizational_Change_Frequency': ('Frecuencia de cambios organizacionales; ventana por validar', 'Histórico / validar'), 'Cross_Functional_Dependencies': ('Dependencias entre áreas', 'Candidato intake'), 'Previous_Delivery_Success_Rate': ('Tasa de éxito de entregas anteriores; corte temporal por validar', 'Histórico / validar'), 'Technical_Debt_Level': ('Nivel de deuda técnica; aplicabilidad por validar', 'Candidato intake'), 'Project_Manager_Experience': ('Experiencia del responsable del proyecto', 'Candidato intake'), 'Org_Process_Maturity': ('Madurez de procesos de la organización', 'Candidato intake'), 'Data_Security_Requirements': ('Exigencias de seguridad de datos', 'Candidato intake'), 'Key_Stakeholder_Availability': ('Disponibilidad de stakeholders clave', 'Candidato intake'), 'Tech_Environment_Stability': ('Estabilidad del entorno tecnológico; N/A fuera de IT', 'Candidato intake'), 'Contract_Type': ('Tipo de contrato', 'Candidato intake'), 'Resource_Contention_Level': ('Competencia por los recursos', 'Candidato intake'), 'Industry_Volatility': ('Volatilidad de la industria', 'Histórico / validar'), 'Client_Experience_Level': ('Experiencia o relación previa del cliente; definición por validar', 'Histórico / validar'), 'Change_Control_Maturity': ('Madurez del control de cambios', 'Candidato intake'), 'Risk_Management_Maturity': ('Madurez de gestión de riesgos', 'Candidato intake'), 'Team_Colocation': ('Modalidad de ubicación del equipo', 'Candidato intake'), 'Documentation_Quality': ('Calidad de documentación; momento de evaluación por validar', 'Posterior / riesgo de fuga'), 'Project_Start_Month': ('Mes de inicio, sin año ni fecha exacta', 'Candidato intake'), 'Current_Phase_Duration_Months': ('Duración de la fase actual en meses', 'Posterior / riesgo de fuga'), 'Seasonal_Risk_Factor': ('Factor de riesgo estacional; fórmula y fecha por validar', 'Histórico / validar'), 'Risk_Level': ('Nivel de riesgo final según el caso', 'Objetivo')}
diccionario = pd.DataFrame([
    {'columna': c, 'descripción propuesta': descripciones[c][0],
     'disponibilidad propuesta': descripciones[c][1], 'tipo': str(df[c].dtype),
     'valores distintos': df[c].nunique(dropna=False),
     'ejemplos': ', '.join(map(str, df[c].drop_duplicates().head(4)))}
    for c in df.columns
])
display(diccionario)


,columna,descripción propuesta,disponibilidad propuesta,tipo,valores distintos,ejemplos
0,Project_ID,Identificador del proyecto,Identificador,str,4000,"PROJ_0001, PROJ_0002, PROJ_0003, PROJ_0004"
1,Project_Type,Tipo o sector del proyecto,Candidato intake,str,6,"Construction, Manufacturing, IT, R&D"
2,Team_Size,Número de integrantes del equipo,Candidato intake,int64,49,"32, 2, 12, 24"
3,Project_Budget_USD,Presupuesto del proyecto en USD,Candidato intake,float64,4000,"1526276.55, 390790.15, 246674.76, 1427830.63"
4,Estimated_Timeline_Months,Duración estimada en meses,Candidato intake,int64,35,"32, 9, 6, 17"
5,Complexity_Score,Puntaje de complejidad; validar escala y quién...,Candidato intake,float64,796,"9.7, 2.72, 2.04, 7.54"
6,Stakeholder_Count,Cantidad de stakeholders,Candidato intake,int64,27,"16, 9, 7, 17"
7,Methodology_Used,Metodología de trabajo,Candidato intake,str,5,"Waterfall, Kanban, Agile, Scrum"
8,Team_Experience_Level,Nivel de experiencia del equipo,Candidato intake,str,4,"Senior, Mixed, Junior, Expert"
9,Past_Similar_Projects,Cantidad de proyectos similares previos,Histórico / validar,int64,11,"3, 0, 1, 2"


## 3. Calidad de los datos
Revisamos vacíos reales, marcadores de texto, duplicados y rangos. `N/A` puede significar “no aplica” y `None` puede ser una categoría de madurez; no los convertimos automáticamente en datos faltantes.


In [5]:
calidad = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'vacíos_o_espacios': df.apply(lambda s: s.astype(str).str.strip().eq('').sum()),
    'N/A_literal': df.eq('N/A').sum(),
    'None_literal': df.eq('None').sum(),
    'valores_distintos': df.nunique(dropna=False)
})
display(calidad)
display(pd.Series({
    'filas duplicadas completas': int(df.duplicated().sum()),
    'IDs duplicados': int(df['Project_ID'].duplicated().sum()),
    'duplicados sin considerar ID': int(df.drop(columns='Project_ID').duplicated().sum()),
    'columnas constantes': int(df.nunique(dropna=False).le(1).sum())
}, name='conteo').to_frame())
display(pd.crosstab(df['Project_Type'], df['Tech_Environment_Stability']))


,tipo,nulos,vacíos_o_espacios,N/A_literal,None_literal,valores_distintos
Project_ID,str,0,0,0,0,4000
Project_Type,str,0,0,0,0,6
Team_Size,int64,0,0,0,0,49
Project_Budget_USD,float64,0,0,0,0,4000
Estimated_Timeline_Months,int64,0,0,0,0,35
Complexity_Score,float64,0,0,0,0,796
Stakeholder_Count,int64,0,0,0,0,27
Methodology_Used,str,0,0,0,0,5
Team_Experience_Level,str,0,0,0,0,4
Past_Similar_Projects,int64,0,0,0,0,11


,conteo
filas duplicadas completas,0
IDs duplicados,0
duplicados sin considerar ID,0
columnas constantes,0


Tech_Environment_Stability,Legacy/Unstable,Mixed,Modern/Stable,N/A
Project_Type,,,,
Construction,0,0,0,797
Healthcare,0,0,0,398
IT,283,542,556,0
Manufacturing,0,0,0,418
Marketing,0,0,0,418
R&D,0,0,0,588


**Lectura inicial:** el marcador `N/A` aparece 2.619 veces (65,48%) en `Tech_Environment_Stability`, exclusivamente en tipos de proyecto distintos de IT. Esto es consistente con “no aplica”, aunque hay que confirmarlo con el dueño del dato. Conviene preservar esa categoría y no imputarla con la moda de IT.


In [6]:
numericas = df.select_dtypes(include='number').columns.tolist()
display(df[numericas].describe(percentiles=[.05, .25, .5, .75, .95]).T)

# Chequeos tentativos: son alertas de revisión, no reglas oficiales del dataset.
tasas_0_1 = ['Team_Turnover_Rate', 'Vendor_Reliability_Score',
             'Resource_Availability', 'Previous_Delivery_Success_Rate']
controles = {f'{c} fuera de [0,1]': int((~df[c].between(0, 1)).sum()) for c in tasas_0_1}
controles.update({
    'mes fuera de 1 a 12': int((~df.Project_Start_Month.between(1, 12)).sum()),
    'presupuesto no positivo': int(df.Project_Budget_USD.le(0).sum()),
    'tamaño del equipo no positivo': int(df.Team_Size.le(0).sum()),
    'plazo estimado no positivo': int(df.Estimated_Timeline_Months.le(0).sum()),
    'uso presupuestario mayor a 1 (revisar, no descartar)': int(df.Budget_Utilization_Rate.gt(1).sum()),
    'fase actual mayor al plazo estimado (revisar)': int(df.Current_Phase_Duration_Months.gt(df.Estimated_Timeline_Months).sum())
})
display(pd.Series(controles, name='registros').to_frame())

# IQR identifica extremos estadísticos, no errores confirmados.
q1, q3 = df[numericas].quantile(.25), df[numericas].quantile(.75)
iqr = q3-q1
extremos = ((df[numericas] < q1-1.5*iqr) | (df[numericas] > q3+1.5*iqr)).sum()
display(extremos.sort_values(ascending=False).head(10).rename('fuera de límites IQR').to_frame())


,count,mean,std,min,5%,25%,50%,75%,95%,max
Team_Size,"4,000.000",15.388,9.221,2.000,4.000,9.000,13.000,20.000,34.000,50.000
Project_Budget_USD,"4,000.000","1,143,031.503","590,878.115","159,355.550","415,680.562","692,532.915","1,007,471.805","1,475,870.195","2,337,685.730","3,768,354.370"
Estimated_Timeline_Months,"4,000.000",17.148,6.927,2.000,7.000,12.000,17.000,22.000,29.000,36.000
Complexity_Score,"4,000.000",6.193,2.213,1.620,2.830,4.460,6.015,7.863,10.000,10.000
Stakeholder_Count,"4,000.000",11.130,4.426,2.000,5.000,8.000,10.000,14.000,19.000,29.000
Past_Similar_Projects,"4,000.000",1.974,1.750,0.000,0.000,1.000,2.000,3.000,5.000,10.000
External_Dependencies_Count,"4,000.000",3.128,1.609,0.000,1.000,2.000,3.000,4.000,6.000,7.000
Change_Request_Frequency,"4,000.000",1.638,1.170,0.010,0.290,0.760,1.370,2.230,3.920,8.840
Team_Turnover_Rate,"4,000.000",0.293,0.167,0.000,0.060,0.160,0.270,0.400,0.600,0.850
Vendor_Reliability_Score,"4,000.000",0.712,0.163,0.090,0.410,0.600,0.730,0.840,0.940,1.000


,registros
"Team_Turnover_Rate fuera de [0,1]",0
"Vendor_Reliability_Score fuera de [0,1]",0
"Resource_Availability fuera de [0,1]",0
"Previous_Delivery_Success_Rate fuera de [0,1]",0
mes fuera de 1 a 12,0
presupuesto no positivo,0
tamaño del equipo no positivo,0
plazo estimado no positivo,0
"uso presupuestario mayor a 1 (revisar, no descartar)",1641
fase actual mayor al plazo estimado (revisar),0


,fuera de límites IQR
Seasonal_Risk_Factor,453
Technical_Debt_Level,427
Schedule_Pressure,340
Historical_Risk_Incidents,263
Organizational_Change_Frequency,158
Team_Size,152
Change_Request_Frequency,117
Communication_Frequency,113
Project_Budget_USD,72
Past_Similar_Projects,64


Un uso presupuestario superior a 1 podría reflejar sobreconsumo; la fórmula no está documentada. No debe descartarse automáticamente. El criterio IQR es poco informativo cuando una variable tiene muchos ceros o muy pocos valores distintos, como puede ocurrir con `Seasonal_Risk_Factor`. No eliminamos extremos ni modificamos el CSV.


## 4. Composición de los proyectos y del riesgo
`Risk_Level` es el resultado que se querría anticipar. Conservamos el orden Low, Medium, High y Critical. La proporción High + Critical es una agrupación exploratoria, no una regla de asignación ya aprobada.


In [7]:
orden_riesgo = ['Low', 'Medium', 'High', 'Critical']
riesgo = df.Risk_Level.value_counts().reindex(orden_riesgo, fill_value=0).to_frame('proyectos')
riesgo['porcentaje'] = riesgo.proyectos / len(df) * 100
display(riesgo)
barras(riesgo.proyectos, 'Proyectos por riesgo final')
alto = df.Risk_Level.isin(['High', 'Critical'])
print(f'High + Critical: {alto.sum():,} proyectos ({alto.mean():.2%})')
for columna in ['Project_Type', 'Project_Phase', 'Methodology_Used']:
    tabla = df[columna].value_counts().to_frame('proyectos')
    tabla['porcentaje'] = tabla.proyectos / len(df) * 100
    print(columna)
    display(tabla)
barras(df.Project_Type.value_counts(), 'Composición por tipo de proyecto')


,proyectos,porcentaje
Risk_Level,,
Low,806,20.150
Medium,1396,34.900
High,1036,25.900
Critical,762,19.050


High + Critical: 1,798 proyectos (44.95%)
Project_Type


,proyectos,porcentaje
Project_Type,,
IT,1381,34.525
Construction,797,19.925
R&D,588,14.700
Manufacturing,418,10.450
Marketing,418,10.450
Healthcare,398,9.950


Project_Phase


,proyectos,porcentaje
Project_Phase,,
Initiation,834,20.850
Planning,813,20.325
Closure,794,19.850
Execution,780,19.500
Monitoring,779,19.475


Methodology_Used


,proyectos,porcentaje
Methodology_Used,,
Agile,1124,28.100
Scrum,1014,25.350
Waterfall,801,20.025
Kanban,575,14.375
Hybrid,486,12.150


**Inconsistencia a aclarar:** el caso describe proyectos cerrados, pero `Project_Phase` contiene Initiation, Planning, Execution, Monitoring y Closure. Podría ser la fase de una captura anterior al cierre, aunque no hay fechas que lo demuestren. Debemos confirmar cuándo se midió cada predictor y cuándo se asignó el riesgo final. `Project_Start_Month` no permite reconstruir cohortes o tendencias cronológicas porque no incluye año.


## 5. Relaciones exploratorias con el riesgo
### Variables numéricas
Usamos correlación de Spearman: correlación de Pearson entre rangos, con promedio para empates. Solo presupone el orden Low < Medium < High < Critical, sin interpretar distancias iguales entre niveles. Los resultados son asociaciones bivariadas, no causas ni importancia predictiva de un modelo. Se calculan sobre todo el archivo como exploración; una evaluación futura deberá reservar datos de prueba.


In [8]:
riesgo_ordinal = df.Risk_Level.map({'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3})
assert riesgo_ordinal.notna().all(), 'Hay etiquetas de riesgo sin mapear.'
correlaciones = df[numericas].rank(method='average').corrwith(riesgo_ordinal.rank(method='average'))
asociaciones = correlaciones.rename('rho_spearman').to_frame()
asociaciones['magnitud'] = asociaciones.rho_spearman.abs()
asociaciones['disponibilidad propuesta'] = [descripciones[c][1] for c in asociaciones.index]
asociaciones = asociaciones.sort_values('magnitud', ascending=False)
display(asociaciones)
barras(asociaciones.head(10).magnitud, 'Diez mayores asociaciones numéricas (valor absoluto de Spearman)')
display(df.groupby('Risk_Level')[['Complexity_Score', 'External_Dependencies_Count',
    'Estimated_Timeline_Months', 'Past_Similar_Projects']].median().reindex(orden_riesgo))


,rho_spearman,magnitud,disponibilidad propuesta
Complexity_Score,0.340,0.340,Candidato intake
Estimated_Timeline_Months,0.283,0.283,Candidato intake
External_Dependencies_Count,0.261,0.261,Candidato intake
Team_Turnover_Rate,0.237,0.237,Posterior / riesgo de fuga
Team_Size,0.198,0.198,Candidato intake
Project_Budget_USD,0.193,0.193,Candidato intake
Stakeholder_Count,0.171,0.171,Candidato intake
Past_Similar_Projects,-0.160,0.160,Histórico / validar
Previous_Delivery_Success_Rate,-0.155,0.155,Histórico / validar
Current_Phase_Duration_Months,0.153,0.153,Posterior / riesgo de fuga


,Complexity_Score,External_Dependencies_Count,Estimated_Timeline_Months,Past_Similar_Projects
Risk_Level,,,,
Low,5.150,3.000,14.000,2.000
Medium,5.560,3.000,15.000,2.000
High,6.370,3.000,18.000,1.000
Critical,7.980,4.000,21.000,1.000


La mayor asociación numérica observada es `Complexity_Score` (ρ ≈ 0,340), seguida de `Estimated_Timeline_Months` (≈ 0,283) y `External_Dependencies_Count` (≈ 0,261). `Team_Turnover_Rate` también se asocia (≈ 0,237), pero podría medirse después del ingreso. Son señales para investigar; por sí solas no permiten decidir cupos ni demostrar que intervenir una variable reduzca el riesgo.


### Variables categóricas
Para comparar categorías sin imponerles un orden, calculamos V de Cramér con corrección de sesgo. Excluimos `Project_ID` y el objetivo. Se muestran además tamaños de grupo: una tasa alta en pocos proyectos puede ser inestable. V de Cramér y Spearman tienen interpretaciones distintas y no deben mezclarse en un único ranking.


In [9]:
def cramer_v_corregido(x, y):
    observada = pd.crosstab(x, y).to_numpy(dtype=float)
    n = observada.sum()
    r, k = observada.shape
    if n <= 1 or min(r, k) <= 1:
        return np.nan
    esperada = np.outer(observada.sum(axis=1), observada.sum(axis=0)) / n
    chi2 = ((observada - esperada)**2 / esperada).sum()
    phi2 = max(0, chi2/n - (k-1)*(r-1)/(n-1))
    r_corregido = r - (r-1)**2/(n-1)
    k_corregido = k - (k-1)**2/(n-1)
    denominador = min(r_corregido-1, k_corregido-1)
    return np.sqrt(phi2/denominador) if denominador > 0 else np.nan

categoricas = [c for c in df.select_dtypes(exclude='number').columns
               if c not in ['Project_ID', 'Risk_Level']]
v_cramer = pd.Series({c: cramer_v_corregido(df[c], df.Risk_Level)
                     for c in categoricas}, name='V_Cramér_corregido').sort_values(ascending=False)
display(v_cramer.to_frame())

def segmento_riesgo(columna):
    tabla = pd.crosstab(df[columna], df.Risk_Level).reindex(columns=orden_riesgo, fill_value=0)
    tabla['proyectos'] = tabla.sum(axis=1)
    tabla['porcentaje High + Critical'] = (tabla.High + tabla.Critical) / tabla.proyectos * 100
    return tabla.sort_values('porcentaje High + Critical', ascending=False)

for columna in list(v_cramer.head(3).index):
    print(columna)
    display(segmento_riesgo(columna))
print('Cruce útil para entender diferencias entre sectores:')
display(segmento_riesgo('Project_Type'))


,V_Cramér_corregido
Org_Process_Maturity,0.234
Technology_Familiarity,0.224
Team_Experience_Level,0.215
Risk_Management_Maturity,0.193
Change_Control_Maturity,0.183
Project_Manager_Experience,0.176
Industry_Volatility,0.161
Resource_Contention_Level,0.159
Key_Stakeholder_Availability,0.154
Stakeholder_Engagement_Level,0.141


Org_Process_Maturity


Risk_Level,Low,Medium,High,Critical,proyectos,porcentaje High + Critical
Org_Process_Maturity,,,,,,
Ad-hoc,24,113,202,255,594,76.936
Defined,167,443,412,245,1267,51.855
Managed,309,529,328,214,1380,39.275
Optimizing,306,311,94,48,759,18.709


Technology_Familiarity


Risk_Level,Low,Medium,High,Critical,proyectos,porcentaje High + Critical
Technology_Familiarity,,,,,,
New,93,286,347,358,1084,65.037
Familiar,355,614,472,311,1752,44.692
Expert,358,496,217,93,1164,26.632


Team_Experience_Level


Risk_Level,Low,Medium,High,Critical,proyectos,porcentaje High + Critical
Team_Experience_Level,,,,,,
Junior,66,158,163,198,585,61.709
Mixed,187,577,487,342,1593,52.040
Senior,321,527,351,208,1407,39.730
Expert,232,134,35,14,415,11.807


Cruce útil para entender diferencias entre sectores:


Risk_Level,Low,Medium,High,Critical,proyectos,porcentaje High + Critical
Project_Type,,,,,,
R&D,38,143,196,211,588,69.218
Construction,163,255,226,153,797,47.553
Marketing,83,153,120,62,418,43.541
IT,305,525,313,238,1381,39.899
Manufacturing,104,158,99,57,418,37.321
Healthcare,113,162,82,41,398,30.905


## 6. Qué datos pediríamos al ingresar
Esta separación ayuda a evitar **fuga de información**: usar datos que solo se conocen cuando el proyecto ya avanzó o cuando se conoce su resultado. No entrenamos un modelo con estas listas todavía. El siguiente paso es contrastarlas con el proceso real y conservar versiones históricas de día 1.


In [10]:
for grupo in ['Candidato intake', 'Histórico / validar', 'Posterior / riesgo de fuga']:
    print(f'\n{grupo}')
    display(diccionario.loc[diccionario['disponibilidad propuesta'].eq(grupo),
                           ['columna', 'descripción propuesta']].reset_index(drop=True))



Candidato intake


,columna,descripción propuesta
0,Project_Type,Tipo o sector del proyecto
1,Team_Size,Número de integrantes del equipo
2,Project_Budget_USD,Presupuesto del proyecto en USD
3,Estimated_Timeline_Months,Duración estimada en meses
4,Complexity_Score,Puntaje de complejidad; validar escala y quién...
5,Stakeholder_Count,Cantidad de stakeholders
6,Methodology_Used,Metodología de trabajo
7,Team_Experience_Level,Nivel de experiencia del equipo
8,External_Dependencies_Count,Número de dependencias externas
9,Requirement_Stability,Estabilidad de los requerimientos



Histórico / validar


,columna,descripción propuesta
0,Past_Similar_Projects,Cantidad de proyectos similares previos
1,Vendor_Reliability_Score,Confiabilidad del proveedor; escala por validar
2,Historical_Risk_Incidents,Incidentes históricos; validar si preceden al ...
3,Market_Volatility,Volatilidad del mercado; escala y fecha por va...
4,Priority_Level,Prioridad del proyecto; validar cómo se decidió
5,Organizational_Change_Frequency,Frecuencia de cambios organizacionales; ventan...
6,Previous_Delivery_Success_Rate,Tasa de éxito de entregas anteriores; corte te...
7,Industry_Volatility,Volatilidad de la industria
8,Client_Experience_Level,Experiencia o relación previa del cliente; def...
9,Seasonal_Risk_Factor,Factor de riesgo estacional; fórmula y fecha p...



Posterior / riesgo de fuga


,columna,descripción propuesta
0,Change_Request_Frequency,Frecuencia de solicitudes de cambio; unidad no...
1,Project_Phase,Fase registrada del proyecto
2,Team_Turnover_Rate,Tasa de rotación del equipo; ventana temporal ...
3,Communication_Frequency,Frecuencia de comunicación; unidad y momento p...
4,Stakeholder_Engagement_Level,Nivel de participación de stakeholders
5,Schedule_Pressure,Presión del cronograma; cálculo por validar
6,Budget_Utilization_Rate,Proporción de utilización presupuestaria
7,Documentation_Quality,Calidad de documentación; momento de evaluació...
8,Current_Phase_Duration_Months,Duración de la fase actual en meses


## 7. Análisis recomendados y orden de trabajo

| Prioridad | Análisis | Pregunta que resuelve | Siguiente paso concreto |
|---|---|---|---|
| 1 | Definición y disponibilidad de datos | ¿Qué se sabe realmente el día 1? | Validar escalas, unidades, fechas de captura, significado de Risk_Level y origen de etiquetas. Resolver las fases de proyectos cerrados. |
| 2 | Riesgo por segmentos | ¿Qué tipos de proyecto, complejidad y dependencias concentran riesgo? | Usar tasas y tamaños de grupo; revisar diferencias dentro de cada sector para evitar confusión por composición. |
| 3 | Baseline de predicción al ingreso | ¿Mejoramos una referencia simple con datos disponibles? | Comparar clase mayoritaria y reglas transparentes con un modelo sencillo. Separar entrenamiento y prueba antes de ajustar transformaciones o elegir reglas. |
| 4 | Calidad de predicción | ¿Detectamos High/Critical y qué errores cometemos? | Medir recall de High/Critical, precisión, matriz de confusión, macro-F1 y calibración. Si existen clientes repetidos, separar por cliente; si aparecen fechas, validar en períodos posteriores. |
| 5 | Capacidad de 50 cupos | ¿A quién atendemos cuando entran 60 proyectos? | Simular lotes mensuales de 40, 50 y 60, claramente identificados como escenarios. Ordenar por riesgo estimado y criterios de negocio acordados. Medir recall y precisión entre los seleccionados, uso de cupos y demanda desbordada. |
| 6 | Diseño del proceso | ¿Cómo convertimos el análisis en una acción? | Explorar un diagnóstico temprano de complejidad y dependencias en Initiation/Planning; definir responsable, handover y escalamiento. Validar la intervención con seguimiento posterior. |

**Límites para los próximos análisis:** no hay fecha completa de ingreso o cierre, año, identificador de cliente, costo de atención, esfuerzo requerido ni resultados de una intervención. No se pueden reconstruir meses reales, calcular ROI de atención ni demostrar causalidad. No sabemos si el dataset es real o sintético. Debe confirmarse su procedencia y cómo se calculó `Risk_Level`: si deriva mecánicamente de los predictores, un buen modelo podría aprender esa fórmula sin anticipar incidentes reales.

**Propuesta para seguir:** comenzar por un formulario breve con tipo de proyecto, tamaño del equipo, presupuesto, duración estimada, complejidad y dependencias. Estos campos son candidatos iniciales por su significado y disponibilidad plausible; todavía hay que validar su captura. Incorporar experiencia y gobernanza solo con definiciones consistentes y una fuente anterior al ingreso.


## 8. Hallazgos iniciales

- El archivo contiene 4.000 proyectos, 51 columnas y cuatro clases de riesgo. Los IDs no se repiten y no se detectan filas completas duplicadas ni campos vacíos.
- Medium representa el 34,9%, High el 25,9%, Low el 20,15% y Critical el 19,05%. High + Critical reúne el 44,95% del histórico. Esta proporción no garantiza la mezcla de proyectos futuros.
- Hay 2.619 valores `N/A` en estabilidad tecnológica, asociados a proyectos fuera de IT; deben tratarse como posible no aplicabilidad.
- Complejidad, duración estimada y dependencias externas son las principales asociaciones numéricas con el riesgo final. Conviene investigarlas con el sector y la disponibilidad temporal de cada campo.
- La prioridad antes de modelar es validar qué valores existían al ingresar. Las columnas de ejecución pueden mejorar una correlación histórica y aun así resultar inadecuadas para el objetivo del caso.

**Rastro de uso de IA de esta etapa:** se utilizó asistencia de IA para estructurar la exploración, proponer un diccionario interpretativo y controles de calidad, generar el código y redactar recomendaciones. Se evitó el comportamiento predeterminado de lectura del CSV que convertiría `N/A` y `None` en nulos sin revisar su significado. Este registro cubre esta exploración; conserva también el prompt o vínculo de esta conversación para el entregable final del caso.


## 9. Validación de las 13 variables candidatas para el formulario

Este apartado responde: **¿qué valores existen realmente y cómo podríamos capturarlos?** Se exploran las cinco variables numéricas y ocho categóricas propuestas en `03_datos_ingreso_y_prediccion.md`. Son candidatos, no una selección predictiva validada.

Los rangos observados describen este histórico; **no son límites de negocio**. Un nuevo presupuesto fuera del rango requiere revisión de extrapolación, no rechazo automático. Las categorías se conservan tal como aparecen en el CSV, antes de traducirlas para el formulario.


In [11]:
variables_formulario = [
    'Project_Type', 'Project_Budget_USD', 'Estimated_Timeline_Months',
    'Team_Size', 'Stakeholder_Count', 'External_Dependencies_Count',
    'Team_Experience_Level', 'Technology_Familiarity', 'Requirement_Stability',
    'Executive_Sponsorship', 'Key_Stakeholder_Availability',
    'Org_Process_Maturity', 'Risk_Management_Maturity'
]
faltantes_esquema = sorted(set(variables_formulario) - set(df.columns))
if faltantes_esquema:
    raise ValueError(f'Columnas no encontradas: {faltantes_esquema}')
datos_formulario = df[variables_formulario].copy()
numericas_formulario = datos_formulario.select_dtypes(include='number').columns.tolist()
categoricas_formulario = datos_formulario.select_dtypes(exclude='number').columns.tolist()

resumen_formulario = pd.DataFrame([
    {'columna': c, 'tipo en pandas': str(df[c].dtype),
     'naturaleza': 'numérica' if c in numericas_formulario else 'categórica',
     'valores distintos': df[c].nunique(dropna=False),
     'nulos reales': int(df[c].isna().sum()),
     'vacíos o espacios': int(df[c].astype(str).str.strip().eq('').sum()),
     'captura propuesta': ('número decimal' if c == 'Project_Budget_USD' else 'número entero')
                          if c in numericas_formulario else 'lista de opciones'}
    for c in variables_formulario
])
print(f'{len(variables_formulario)} variables: {len(numericas_formulario)} numéricas y '
      f'{len(categoricas_formulario)} categóricas.')
display(resumen_formulario)


13 variables: 5 numéricas y 8 categóricas.


,columna,tipo en pandas,naturaleza,valores distintos,nulos reales,vacíos o espacios,captura propuesta
0,Project_Type,str,categórica,6,0,0,lista de opciones
1,Project_Budget_USD,float64,numérica,4000,0,0,número decimal
2,Estimated_Timeline_Months,int64,numérica,35,0,0,número entero
3,Team_Size,int64,numérica,49,0,0,número entero
4,Stakeholder_Count,int64,numérica,27,0,0,número entero
5,External_Dependencies_Count,int64,numérica,8,0,0,número entero
6,Team_Experience_Level,str,categórica,4,0,0,lista de opciones
7,Technology_Familiarity,str,categórica,3,0,0,lista de opciones
8,Requirement_Stability,str,categórica,3,0,0,lista de opciones
9,Executive_Sponsorship,str,categórica,3,0,0,lista de opciones


### 9.1. Rangos y variantes numéricas
Mostramos mínimos, máximos, percentiles, cantidad de valores distintos y presencia de decimales. Las columnas almacenadas como números pueden representar cantidades discretas: siguen siendo numéricas y no requieren convertirse en listas de categorías.

El número de valores distintos del presupuesto no significa que el formulario deba ofrecer todas esas opciones.


In [12]:
unidades_formulario = {
    'Project_Budget_USD': 'USD', 'Estimated_Timeline_Months': 'meses',
    'Team_Size': 'personas', 'Stakeholder_Count': 'stakeholders',
    'External_Dependencies_Count': 'dependencias externas'
}
perfil_numerico = pd.DataFrame([
    {'columna': c, 'unidad propuesta': unidades_formulario[c],
     'mínimo observado': df[c].min(), 'P25': df[c].quantile(.25),
     'mediana': df[c].median(), 'P75': df[c].quantile(.75),
     'máximo observado': df[c].max(),
     'valores distintos': df[c].nunique(),
     'registros con decimales': int(df[c].dropna().mod(1).ne(0).sum()),
     'registros en cero': int(df[c].eq(0).sum())}
    for c in numericas_formulario
]).set_index('columna')
display(perfil_numerico)

# Mostrar todos los valores de las cantidades discretas, con su frecuencia.
frecuencias_discretas = []
for c in numericas_formulario:
    if c == 'Project_Budget_USD':
        continue
    frecuencias = df[c].value_counts(dropna=False).sort_index()
    frecuencias_discretas.extend([
        {'columna': c, 'valor observado': valor, 'proyectos': int(n),
         'porcentaje': n/len(df)*100}
        for valor, n in frecuencias.items()
    ])
frecuencias_discretas = pd.DataFrame(frecuencias_discretas)
for c in numericas_formulario:
    if c != 'Project_Budget_USD':
        display(frecuencias_discretas.loc[frecuencias_discretas.columna.eq(c)].reset_index(drop=True))


,unidad propuesta,mínimo observado,P25,mediana,P75,máximo observado,valores distintos,registros con decimales,registros en cero
columna,,,,,,,,,
Project_Budget_USD,USD,"159,355.550","692,532.915","1,007,471.805","1,475,870.195","3,768,354.370",4000,3965,0
Estimated_Timeline_Months,meses,2.000,12.000,17.000,22.000,36.000,35,0,0
Team_Size,personas,2.000,9.000,13.000,20.000,50.000,49,0,0
Stakeholder_Count,stakeholders,2.000,8.000,10.000,14.000,29.000,27,0,0
External_Dependencies_Count,dependencias externas,0.000,2.000,3.000,4.000,7.000,8,0,176


,columna,valor observado,proyectos,porcentaje
0,Estimated_Timeline_Months,2,1,0.025
1,Estimated_Timeline_Months,3,14,0.350
2,Estimated_Timeline_Months,4,24,0.600
3,Estimated_Timeline_Months,5,52,1.300
4,Estimated_Timeline_Months,6,84,2.100
5,Estimated_Timeline_Months,7,117,2.925
6,Estimated_Timeline_Months,8,142,3.550
7,Estimated_Timeline_Months,9,162,4.050
8,Estimated_Timeline_Months,10,174,4.350
9,Estimated_Timeline_Months,11,204,5.100


,columna,valor observado,proyectos,porcentaje
0,Team_Size,2,82,2.050
1,Team_Size,3,61,1.525
2,Team_Size,4,91,2.275
3,Team_Size,5,122,3.050
4,Team_Size,6,156,3.900
5,Team_Size,7,203,5.075
6,Team_Size,8,220,5.500
7,Team_Size,9,244,6.100
8,Team_Size,10,254,6.350
9,Team_Size,11,223,5.575


,columna,valor observado,proyectos,porcentaje
0,Stakeholder_Count,2,7,0.175
1,Stakeholder_Count,3,35,0.875
2,Stakeholder_Count,4,98,2.450
3,Stakeholder_Count,5,193,4.825
4,Stakeholder_Count,6,205,5.125
5,Stakeholder_Count,7,297,7.425
6,Stakeholder_Count,8,369,9.225
7,Stakeholder_Count,9,399,9.975
8,Stakeholder_Count,10,405,10.125
9,Stakeholder_Count,11,345,8.625


,columna,valor observado,proyectos,porcentaje
0,External_Dependencies_Count,0,176,4.400
1,External_Dependencies_Count,1,491,12.275
2,External_Dependencies_Count,2,780,19.500
3,External_Dependencies_Count,3,943,23.575
4,External_Dependencies_Count,4,801,20.025
5,External_Dependencies_Count,5,500,12.500
6,External_Dependencies_Count,6,236,5.900
7,External_Dependencies_Count,7,73,1.825


### 9.2. Catálogo completo de categorías y frecuencias
Se muestra **cada variante observada**, sin limitarse a ejemplos, con número de proyectos y porcentaje de las 4.000 filas. También se comprueba si diferencias de espacios o mayúsculas podrían estar generando variantes artificiales. Este diagnóstico no modifica los valores originales.


In [13]:
catalogo_formulario = {}
filas_categorias = []
for c in categoricas_formulario:
    conteos = df[c].value_counts(dropna=False).sort_index()
    catalogo_formulario[c] = sorted(df[c].dropna().unique().tolist())
    for categoria, n in conteos.items():
        filas_categorias.append({'columna': c, 'categoría exacta en CSV': categoria,
                                'proyectos': int(n), 'porcentaje': n/len(df)*100})
catalogo_frecuencias = pd.DataFrame(filas_categorias)
display(pd.DataFrame([
    {'columna': c, 'cantidad de categorías': len(opciones),
     'todas las categorías observadas': ' / '.join(opciones)}
    for c, opciones in catalogo_formulario.items()
]))
for c in categoricas_formulario:
    display(catalogo_frecuencias.loc[catalogo_frecuencias.columna.eq(c)].reset_index(drop=True))

control_variantes = pd.DataFrame([
    {'columna': c, 'variantes originales': df[c].nunique(),
     'variantes al quitar espacios e ignorar mayúsculas':
         df[c].astype(str).str.strip().str.casefold().nunique(),
     'registros con espacios externos': int(df[c].astype(str).ne(df[c].astype(str).str.strip()).sum())}
    for c in categoricas_formulario
])
display(control_variantes)


,columna,cantidad de categorías,todas las categorías observadas
0,Project_Type,6,Construction / Healthcare / IT / Manufacturing...
1,Team_Experience_Level,4,Expert / Junior / Mixed / Senior
2,Technology_Familiarity,3,Expert / Familiar / New
3,Requirement_Stability,3,Moderate / Stable / Volatile
4,Executive_Sponsorship,3,Moderate / Strong / Weak
5,Key_Stakeholder_Availability,5,Excellent / Good / Limited / Moderate / Poor
6,Org_Process_Maturity,4,Ad-hoc / Defined / Managed / Optimizing
7,Risk_Management_Maturity,4,Advanced / Basic / Formal / None


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Project_Type,Construction,797,19.925
1,Project_Type,Healthcare,398,9.950
2,Project_Type,IT,1381,34.525
3,Project_Type,Manufacturing,418,10.450
4,Project_Type,Marketing,418,10.450
5,Project_Type,R&D,588,14.700


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Team_Experience_Level,Expert,415,10.375
1,Team_Experience_Level,Junior,585,14.625
2,Team_Experience_Level,Mixed,1593,39.825
3,Team_Experience_Level,Senior,1407,35.175


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Technology_Familiarity,Expert,1164,29.100
1,Technology_Familiarity,Familiar,1752,43.800
2,Technology_Familiarity,New,1084,27.100


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Requirement_Stability,Moderate,2063,51.575
1,Requirement_Stability,Stable,1173,29.325
2,Requirement_Stability,Volatile,764,19.100


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Executive_Sponsorship,Moderate,1983,49.575
1,Executive_Sponsorship,Strong,1213,30.325
2,Executive_Sponsorship,Weak,804,20.100


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Key_Stakeholder_Availability,Excellent,590,14.750
1,Key_Stakeholder_Availability,Good,833,20.825
2,Key_Stakeholder_Availability,Limited,937,23.425
3,Key_Stakeholder_Availability,Moderate,1082,27.050
4,Key_Stakeholder_Availability,Poor,558,13.950


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Org_Process_Maturity,Ad-hoc,594,14.850
1,Org_Process_Maturity,Defined,1267,31.675
2,Org_Process_Maturity,Managed,1380,34.500
3,Org_Process_Maturity,Optimizing,759,18.975


,columna,categoría exacta en CSV,proyectos,porcentaje
0,Risk_Management_Maturity,Advanced,543,13.575
1,Risk_Management_Maturity,Basic,1334,33.350
2,Risk_Management_Maturity,Formal,1332,33.300
3,Risk_Management_Maturity,None,791,19.775


,columna,variantes originales,variantes al quitar espacios e ignorar mayúsculas,registros con espacios externos
0,Project_Type,6,6,0
1,Team_Experience_Level,4,4,0
2,Technology_Familiarity,3,3,0
3,Requirement_Stability,3,3,0
4,Executive_Sponsorship,3,3,0
5,Key_Stakeholder_Availability,5,5,0
6,Org_Process_Maturity,4,4,0
7,Risk_Management_Maturity,4,4,0


### 9.3. ¿Las opciones propuestas existen en el dataset?
Contrastamos el catálogo sugerido para las ocho listas con las categorías exactas observadas. Una coincidencia confirma la presencia de las opciones, **no su definición ni su disponibilidad el día 1**. Para el formulario se pueden traducir las etiquetas, conservando estos códigos internamente.


In [14]:
opciones_propuestas = {
    'Project_Type': ['Construction', 'Healthcare', 'IT', 'Manufacturing', 'Marketing', 'R&D'],
    'Team_Experience_Level': ['Junior', 'Senior', 'Expert', 'Mixed'],
    'Technology_Familiarity': ['New', 'Familiar', 'Expert'],
    'Requirement_Stability': ['Stable', 'Moderate', 'Volatile'],
    'Executive_Sponsorship': ['Weak', 'Moderate', 'Strong'],
    'Key_Stakeholder_Availability': ['Poor', 'Limited', 'Moderate', 'Good', 'Excellent'],
    'Org_Process_Maturity': ['Ad-hoc', 'Defined', 'Managed', 'Optimizing'],
    'Risk_Management_Maturity': ['None', 'Basic', 'Formal', 'Advanced']
}
validacion_opciones = []
for c, opciones in opciones_propuestas.items():
    observadas = set(catalogo_formulario[c])
    no_observadas = sorted(set(opciones) - observadas)
    no_propuestas = sorted(observadas - set(opciones))
    validacion_opciones.append({
        'columna': c, 'opciones propuestas ausentes del CSV': ', '.join(no_observadas) or 'Ninguna',
        'categorías del CSV omitidas en la propuesta': ', '.join(no_propuestas) or 'Ninguna',
        'catálogo coincide': not no_observadas and not no_propuestas
    })
display(pd.DataFrame(validacion_opciones))

# Controles de integridad de los resúmenes: cada tabla debe sumar todo el dataset.
assert len(variables_formulario) == 13
assert len(numericas_formulario) == 5 and len(categoricas_formulario) == 8
assert catalogo_frecuencias.groupby('columna').proyectos.sum().eq(len(df)).all()
assert frecuencias_discretas.groupby('columna').proyectos.sum().eq(len(df)).all()
assert pd.DataFrame(validacion_opciones)['catálogo coincide'].all()
print('Las ocho listas propuestas coinciden con las categorías observadas en este archivo.')


,columna,opciones propuestas ausentes del CSV,categorías del CSV omitidas en la propuesta,catálogo coincide
0,Project_Type,Ninguna,Ninguna,True
1,Team_Experience_Level,Ninguna,Ninguna,True
2,Technology_Familiarity,Ninguna,Ninguna,True
3,Requirement_Stability,Ninguna,Ninguna,True
4,Executive_Sponsorship,Ninguna,Ninguna,True
5,Key_Stakeholder_Availability,Ninguna,Ninguna,True
6,Org_Process_Maturity,Ninguna,Ninguna,True
7,Risk_Management_Maturity,Ninguna,Ninguna,True


Las ocho listas propuestas coinciden con las categorías observadas en este archivo.


### 9.4. Implicaciones para el formulario y el modelo
- Las cinco cantidades se capturan como números: el presupuesto admite decimales; las otras cuatro tienen semántica de conteo o meses enteros en este archivo.
- Las ocho variables de texto se capturan mediante listas, usando el catálogo completo anterior. En experiencia del equipo existen efectivamente `Junior`, `Senior`, `Expert` y `Mixed`.
- `None` en madurez de gestión de riesgos es una categoría observada; no debe perderse al leer el CSV ni confundirse con desconocido.
- Un rango observado no demuestra que todos sus valores intermedios existan; las tablas de frecuencia permiten comprobarlo para las cantidades discretas.
- Si se permite “Por confirmar”, debe tratarse como información pendiente. Esa opción no se convierte automáticamente en una categoría aprendida por el modelo.
- Este perfil valida el contenido de las columnas. Su disponibilidad al ingreso, sus definiciones y su utilidad predictiva todavía deben validarse por separado.

Este apartado amplía la propuesta inicial del notebook. El diseño vigente evalúa un proyecto y recomienda una acción al ingresar; la discusión anterior sobre cupos es contexto operativo complementario.
